In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")

# Загрузка данных и формирование датасета

In [ ]:
df = pd.read_csv("./data/solo.csv")
df.info()

In [3]:
dataframes = {}
for i in range(1,10):
    cols = ["DATE", f"HRWSPD_WT{i}", f"HRWD_SIN_WT{i}", f"HRWD_COS_WT{i}", f"NSPG_WT{i}"]
    dataframes[f"WT{i}"] = df[cols]

In [4]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score


def metric_row(name: str, y_true: pd.Series, y_pred: np.ndarray | pd.Series) -> dict:
    y_true = pd.Series(np.asarray(y_true), index=y_true.index if hasattr(y_true, "index") else None, dtype=float)
    pred = pd.Series(np.asarray(y_pred), index=y_true.index, dtype=float)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred),
    }

    
def make_data(df, sample_number, test_size, valid_from_train):
    
    test_size = test_size
    valid_from_train = valid_from_train

    n = len(df)
    feature_columns = [f"HRWSPD_WT{sample_number}", f"HRWD_SIN_WT{sample_number}", f"HRWD_COS_WT{sample_number}"]
    target_col = f"NSPG_WT{sample_number}" 
    test_start = int(n * (1 - test_size)) if isinstance(test_size, float) else n - int(test_size)
    train_valid = df.iloc[:test_start].copy()
    test = df.iloc[test_start:].copy()
    
    valid_start = int(len(train_valid) * (1 - valid_from_train))
    train = train_valid.iloc[:valid_start].copy()
    valid = train_valid.iloc[valid_start:].copy()
    
    X_train, y_train = train[feature_columns], train[target_col]
    X_valid, y_valid = valid[feature_columns], valid[target_col]
    X_test, y_test = test[feature_columns], test[target_col]
    
    print(f"Train: {train.index.min()} -> {train.index.max()} ({len(train):,})")
    print(f"Valid: {valid.index.min()} -> {valid.index.max()} ({len(valid):,})")
    print(f"Test:  {test.index.min()} -> {test.index.max()} ({len(test):,})")
    return X_train, y_train, X_valid, y_valid, X_test, y_test


# Обучение моделей

In [ ]:
models = {}
plots_data = {}
metrics=[]
RANDOM_STATE=42
for i in range(1,10):
    print("="*20, f"  Making Dataset for WT{i}", "="*20)
    X_train, y_train, X_valid, y_valid, X_test, y_test = make_data(dataframes[f"WT{i}"], i, 0.2, 0.2)

    print("="*20, f"Training XGBoost for WT{i}", "="*20)
    xgb_model = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.025,
        max_depth=4,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.5,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="rmse",
        early_stopping_rounds=80,
    )
    
    xgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=100,
    )
    
    valid_pred = xgb_model.predict(X_valid)
    test_pred = xgb_model.predict(X_test)

    plots_data[f"y_test_WT{i}"] = y_test
    plots_data[f"test_pred_WT{i}"] = test_pred
    
    models[f"WT{i}"] = xgb_model
    print("="*21, f" Saved XGBoost for WT{i}", "="*21)

    metrics.extend([
        metric_row(f"xgboost_valid_WT{i}", y_valid, valid_pred),
        metric_row(f"xgboost_test_WT{i}", y_test, test_pred),
    ])

metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(10))

====================   Making Dataset for WT1 ====================
Train: 0 -> 11227 (11,228)
Valid: 11228 -> 14034 (2,807)
Test:  14035 -> 17543 (3,509)
==================== Training XGBoost for WT1 ====================
[0]	validation_0-rmse:1131.39320
[100]	validation_0-rmse:299.42815
[200]	validation_0-rmse:237.74082
[297]	validation_0-rmse:238.50948
=====================  Saved XGBoost for WT1 =====================
====================   Making Dataset for WT2 ====================
Train: 0 -> 11227 (11,228)
Valid: 11228 -> 14034 (2,807)
Test:  14035 -> 17543 (3,509)
==================== Training XGBoost for WT2 ====================
[0]	validation_0-rmse:1140.77480
[100]	validation_0-rmse:315.43573
[200]	validation_0-rmse:265.13154
[285]	validation_0-rmse:266.68063
=====================  Saved XGBoost for WT2 =====================
====================   Making Dataset for WT3 ====================
Train: 0 -> 11227 (11,228)
Valid: 11228 -> 14034 (2,807)
Test:  14035 -> 17543 (3,509)


,model,MAE,RMSE,R2
0,xgboost_valid_WT1,100.860700,237.573113,0.956391
1,xgboost_test_WT1,108.035851,204.466895,0.964570
2,xgboost_valid_WT2,111.045354,264.983156,0.946430
3,xgboost_test_WT2,117.096887,214.609033,0.963402
4,xgboost_valid_WT3,105.033288,233.252624,0.959664
5,xgboost_test_WT3,114.783455,200.408719,0.969473
6,xgboost_valid_WT4,109.200389,259.632826,0.949654
7,xgboost_test_WT4,115.891772,211.721293,0.966850
8,xgboost_valid_WT5,114.782535,228.581117,0.957933
9,xgboost_test_WT5,133.184400,210.355768,0.964019


# Оптимизация гиперпараметров

In [6]:
import optuna
from sklearn.metrics import mean_absolute_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

models = {}
plots_data = {}
metrics = []
RANDOM_STATE = 42
N_TRIALS = 500  

for i in range(1, 10):
    print("=" * 20, f"  Making Dataset for WT{i}", "=" * 20)
    X_train, y_train, X_valid, y_valid, X_test, y_test = make_data(
        dataframes[f"WT{i}"], i, 0.2, 0.2
    )

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 500, 4000),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
            "objective": "reg:squarederror",
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "eval_metric": "rmse",
            "early_stopping_rounds": 80,
        }
        model = XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False,
        )
        pred = model.predict(X_valid)
        return mean_absolute_error(y_valid, pred)

    print("=" * 20, f" Optuna HPO for WT{i}", "=" * 20)
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    best_params.update({
        "objective": "reg:absoluteerror",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "eval_metric": "mae",
        "early_stopping_rounds": 80,
    })
    print(f"WT{i} best MAE={study.best_value:.4f} | params: {best_params}")

    print("=" * 20, f" Training final model for WT{i}", "=" * 20)
    xgb_model = XGBRegressor(**best_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=100,
    )

    valid_pred = xgb_model.predict(X_valid)
    test_pred = xgb_model.predict(X_test)

    plots_data[f"y_test_WT{i}"] = y_test
    plots_data[f"test_pred_WT{i}"] = test_pred
    models[f"WT{i}"] = xgb_model

    metrics.extend([
        metric_row(f"xgboost_valid_WT{i}", y_valid, valid_pred),
        metric_row(f"xgboost_test_WT{i}", y_test, test_pred),
    ])

metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(10))

====================   Making Dataset for WT1 ====================
Train: 0 -> 11227 (11,228)
Valid: 11228 -> 14034 (2,807)
Test:  14035 -> 17543 (3,509)
====================  Optuna HPO for WT1 ====================


  0%|          | 0/500 [00:00<?, ?it/s]

WT1 best MAE=95.7465 | params: {'n_estimators': 631, 'learning_rate': 0.013829692302994183, 'max_depth': 3, 'min_child_weight': 7, 'subsample': 0.6130080956841341, 'colsample_bytree': 0.9866483934893137, 'reg_alpha': 0.30233603794464675, 'reg_lambda': 2.19934255036443, 'objective': 'reg:absoluteerror', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1, 'eval_metric': 'mae', 'early_stopping_rounds': 80}
====================  Training final model for WT1 ====================
[0]	validation_0-mae:875.42986
[100]	validation_0-mae:329.01348
[200]	validation_0-mae:140.14233
[300]	validation_0-mae:99.65769
[400]	validation_0-mae:93.88747
[500]	validation_0-mae:93.52172
[540]	validation_0-mae:93.76884
====================   Making Dataset for WT2 ====================
Train: 0 -> 11227 (11,228)
Valid: 11228 -> 14034 (2,807)
Test:  14035 -> 17543 (3,509)
====================  Optuna HPO for WT2 ====================


  0%|          | 0/500 [00:00<?, ?it/s]

WT2 best MAE=106.5235 | params: {'n_estimators': 3304, 'learning_rate': 0.03765538730386452, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 0.8256060108514884, 'colsample_bytree': 0.8359174822035171, 'reg_alpha': 0.837347307999316, 'reg_lambda': 2.2357199588657815, 'objective': 'reg:absoluteerror', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1, 'eval_metric': 'mae', 'early_stopping_rounds': 80}
====================  Training final model for WT2 ====================
[0]	validation_0-mae:862.79351
[100]	validation_0-mae:117.60082
[200]	validation_0-mae:105.57615
[247]	validation_0-mae:105.43192
====================   Making Dataset for WT3 ====================
Train: 0 -> 11227 (11,228)
Valid: 11228 -> 14034 (2,807)
Test:  14035 -> 17543 (3,509)
====================  Optuna HPO for WT3 ====================


...

  0%|          | 0/500 [00:00<?, ?it/s]

WT8 best MAE=107.7610 | params: {'n_estimators': 2687, 'learning_rate': 0.01282346187418458, 'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.627944449938091, 'colsample_bytree': 0.8385343770570854, 'reg_alpha': 0.00011508050440162716, 'reg_lambda': 2.647958391888791, 'objective': 'reg:absoluteerror', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1, 'eval_metric': 'mae', 'early_stopping_rounds': 80}
====================  Training final model for WT8 ====================
[0]	validation_0-mae:850.18028
[100]	validation_0-mae:351.06892
[200]	validation_0-mae:161.35127
[300]	validation_0-mae:111.68802
[400]	validation_0-mae:103.28454
[500]	validation_0-mae:102.66962
[542]	validation_0-mae:103.00138
====================   Making Dataset for WT9 ====================
Train: 0 -> 11227 (11,228)
Valid: 11228 -> 14034 (2,807)
Test:  14035 -> 17543 (3,509)
====================  Optuna HPO for WT9 ====================


  0%|          | 0/500 [00:00<?, ?it/s]

WT9 best MAE=109.6687 | params: {'n_estimators': 2464, 'learning_rate': 0.05555363438814141, 'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.6926795977096366, 'colsample_bytree': 0.8431827639564755, 'reg_alpha': 0.034441815508461586, 'reg_lambda': 2.0875627277798916, 'objective': 'reg:absoluteerror', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1, 'eval_metric': 'mae', 'early_stopping_rounds': 80}
====================  Training final model for WT9 ====================
[0]	validation_0-mae:806.05059
[100]	validation_0-mae:109.77171
[200]	validation_0-mae:108.38765
[300]	validation_0-mae:107.02717
[400]	validation_0-mae:106.55082
[500]	validation_0-mae:106.14856
[599]	validation_0-mae:106.13979


,model,MAE,RMSE,R2
0,xgboost_valid_WT1,93.375768,243.305696,0.954261
1,xgboost_test_WT1,92.889423,203.530875,0.964893
2,xgboost_valid_WT2,104.544736,272.807637,0.943219
3,xgboost_test_WT2,100.783890,214.253842,0.963523
4,xgboost_valid_WT3,92.275320,236.853617,0.958409
5,xgboost_test_WT3,93.632000,192.935830,0.971707
6,xgboost_valid_WT4,101.832300,265.774141,0.947244
7,xgboost_test_WT4,101.124878,209.005253,0.967695
8,xgboost_valid_WT5,107.262941,233.549453,0.956085
9,xgboost_test_WT5,111.159227,199.762578,0.967552


# Визуализация

In [ ]:
for i in range(1,10):
    y_test = plots_data[f"y_test_WT{i}"] 
    test_pred = plots_data[f"test_pred_WT{i}"]
    test_pred_series = pd.Series(test_pred, index=y_test.index, name="prediction")
    plt.figure(figsize=(14, 4), dpi=500)
    
    plt.plot(y_test, label='fact')
    plt.plot(test_pred_series, label='xgboost')
    
    plt.title(f'Параметр NSPG_WT{i}: Оригинал | Предсказание')
    plt.legend(loc='upper right')
    
    plt.show()

# Инференс по всему датасету

In [ ]:
inference_dataframes = {}
total_df = df["DATE"]
for i in range(1,10):
    cols = [f"HRWSPD_WT{i}", f"HRWD_SIN_WT{i}", f"HRWD_COS_WT{i}"]
    inference_dataframes[f"WT{i}"] = df[cols]
    # print(inference_dataframes[f"WT{i}"].columns)
    inference_dataframes[f"WT{i}"][f"NSPG_WT{i}"] = models[f"WT{i}"].predict(inference_dataframes[f"WT{i}"])
    print(inference_dataframes[f"WT{i}"].columns)
    
    total_df = pd.concat([total_df, inference_dataframes[f"WT{i}"]], axis=1)

total_df["Target"] = df["Target"]

In [10]:
total_df.to_csv("data/predicted.csv", index=False)